# Serenity AI — Evaluation Notebook

End-to-end evaluation of Serenity AI agents using [DeepEval](https://docs.confident-ai.com/).

**Workflow:**
1. Fill in **Configuration** (cell below)
2. Run **Setup** to load helpers
3. Run **Load Datasets** to see available test suites
4. Pick a dataset and run **Evaluate**
5. Inspect the **Results** table

---

## 0 · Install dependencies

Run once, then comment out.

In [ ]:
# !pip install -r requirements.txt

## 1 · Configuration

Edit the variables below before running anything else.

In [ ]:
AI_SERVICE_URL     = "http://localhost:8000"
INTERNAL_API_TOKEN = "dev-internal-token"

OPENAI_API_KEY = "sk-..."

AUTH_TOKEN   = ""
ORG_ID       = ""
ORG_SLUG     = ""
ORG_NAME     = ""
USER_ID      = ""
DISPLAY_NAME = ""
EMAIL        = ""

## 2 · Setup

Imports, AI adapter, metric builder, evaluation runner, and results renderer.
Run this cell once — re-run if you change **Configuration**.

In [ ]:
import json
import os
import time
from pathlib import Path
from uuid import uuid4

import httpx
from IPython.display import HTML, display
from tqdm.notebook import tqdm

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from deepeval import evaluate
from deepeval.evaluate.configs import CacheConfig, DisplayConfig
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    FaithfulnessMetric,
    GEval,
)
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


FEATURE_DEFAULT_METRICS: dict[str, list[str]] = {
    "workspace_qa":           ["answer_relevancy", "faithfulness", "contextual_recall"],
    "task_creator":           ["answer_relevancy", "extraction_accuracy"],
    "meeting_scheduler":      ["answer_relevancy", "extraction_accuracy"],
    "document_understanding": ["answer_relevancy", "faithfulness", "contextual_recall"],
    "wiki_editor":            ["answer_relevancy", "extraction_accuracy"],
    "chat_assistant":         ["answer_relevancy"],
}


def _headers() -> dict:
    h = {"Content-Type": "application/json"}
    if INTERNAL_API_TOKEN:
        h["x-internal-api-token"] = INTERNAL_API_TOKEN
    if AUTH_TOKEN:
        h["Authorization"] = AUTH_TOKEN if AUTH_TOKEN.startswith("Bearer ") else f"Bearer {AUTH_TOKEN}"
    return h


def _auth_ctx() -> dict:
    return {
        "orgId": ORG_ID,
        "userId": USER_ID,
        "orgSlug": ORG_SLUG,
        "orgName": ORG_NAME,
        "displayName": DISPLAY_NAME,
        "email": EMAIL,
    }


def call_ai(user_input: str, feature: str, case: dict, session_id: str) -> tuple[str, int]:
    """Call the Serenity AI service for the given feature. Returns (output, latency_ms)."""
    headers = _headers()
    auth    = _auth_ctx()

    if feature == "wiki_editor":
        url     = f"{AI_SERVICE_URL}/api/internal/v1/ai/wiki/edit"
        payload = {
            "authContext": auth,
            "pageId": case.get("page_id", "eval-page"),
            "pageTitle": case.get("page_title", "Eval Page"),
            "pageContentMarkdown": case.get("page_content_markdown", ""),
            "prompt": user_input,
            "orgSlug": ORG_SLUG,
        }
    elif feature == "chat_assistant":
        url     = f"{AI_SERVICE_URL}/api/internal/v1/ai/chat/assist"
        payload = {
            "authContext": auth,
            "conversationContext": case.get("conversation_context", []),
            "prompt": user_input,
        }
    else:
        url = (
            f"{AI_SERVICE_URL}/api/internal/v1/ai/chat"
            if INTERNAL_API_TOKEN
            else f"{AI_SERVICE_URL}/api/ai/chat"
        )
        payload = {
            "sessionId": session_id,
            "messages": [{"role": "user", "content": user_input}],
            "authContext": auth,
            "context": {},
        }

    t0 = time.monotonic()
    with httpx.Client(timeout=120.0) as client:
        resp = client.post(url, json=payload, headers=headers)
        resp.raise_for_status()
    latency_ms = int((time.monotonic() - t0) * 1000)

    data = resp.json()
    if feature == "wiki_editor":
        output = data.get("updatedContentMarkdown") or data.get("answer", "")
    elif feature == "chat_assistant":
        output = data.get("suggestedContent", "")
    else:
        output = data.get("answer", "")

    return output, latency_ms

_GEVAL_SPECS: dict[str, tuple] = {
    "task_creator": (
        "Task Creation Accuracy",
        (
            "Evaluate whether the actual output correctly handles the task creation request from the "
            "input. The response should show the system understood the intent and extracted the key "
            "details: task title, due date if provided, assignee if named, and priority if stated. "
            "Resolving relative dates to concrete dates is correct and must NOT be penalized. "
            "Penalize only when a key detail explicitly stated in the input is missing or factually wrong."
        ),
        [
            "Identify key task details from the input: title, due date, assignee, priority.",
            "Check if the output captures the task title — accept minor paraphrasing.",
            "If a due date is specified, verify the output's date is consistent with the input.",
            "If assignee or priority is named, verify the output includes them.",
            "Verify the output confirms task creation or shows the intent was understood.",
            "Assign a high score when all key details are present and correct.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "meeting_scheduler": (
        "Meeting Scheduling Accuracy",
        (
            "Evaluate whether the actual output correctly handles the meeting or room booking request. "
            "The response should demonstrate that the system understood the booking intent and extracted "
            "the correct details (subject, date, time, room or attendees when explicitly specified). "
            "Relative date expressions are correct as long as the output interprets them consistently."
        ),
        [
            "Identify key scheduling details: subject, date, time, room, attendees.",
            "Check that the output captures the meeting subject correctly.",
            "Verify date and time are consistent with the input request.",
            "If attendees or room are named, verify the output includes them.",
            "Verify the output confirms the booking or shows the intent was understood.",
            "Assign a high score when all key details are present and correct.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
    "wiki_editor": (
        "Wiki Edit Accuracy",
        (
            "Evaluate whether the actual output fulfills the edit request from the input. "
            "The expected output is a reference — the actual output does NOT need to match it exactly. "
            "Focus on whether the edit intent was correctly applied. Different valid markdown structures "
            "are acceptable as long as the factual content is preserved and the edit goal is achieved. "
            "Penalize only when the output ignores the request or loses important information."
        ),
        [
            "Identify the edit goal from the input (reformat, translate, improve clarity, etc.).",
            "Check that the output attempts to fulfill that specific edit goal.",
            "Verify key facts and information from the original content are preserved.",
            "Accept any valid markdown structure that achieves the edit goal.",
            "For translations, verify the target language and meaning are correct.",
            "Assign a high score if the edit goal is achieved and facts are preserved.",
        ],
        [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    ),
}


def build_metrics(metric_names: list[str], feature: str | None = None) -> list:
    metrics = []
    for name in metric_names:
        if name == "answer_relevancy":
            metrics.append(AnswerRelevancyMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "faithfulness":
            metrics.append(FaithfulnessMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "contextual_recall":
            metrics.append(ContextualRecallMetric(threshold=0.3, model="gpt-4o-mini"))
        elif name == "extraction_accuracy":
            if feature in _GEVAL_SPECS:
                g_name, criteria, steps, params = _GEVAL_SPECS[feature]
            else:
                g_name  = "Extraction Accuracy"
                criteria = "Verify if the key information, entities, and actions are accurately extracted from the expected output and represented in the actual output."
                steps   = [
                    "Check whether the actual output correctly identifies all key entities and data points.",
                    "Ensure no incorrect or fabricated information is extracted.",
                    "Verify that the formatting of the extracted data matches the expected structure.",
                ]
                params  = [LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
            metrics.append(GEval(
                name=g_name,
                criteria=criteria,
                evaluation_steps=steps,
                evaluation_params=params,
                threshold=0.3,
                model="gpt-4o-mini",
            ))
    return metrics


def run_evaluation(
    dataset: dict,
    metric_names: list[str] | None = None,
) -> list[dict]:
    """Run DeepEval metrics on every case in a dataset. Returns a list of result dicts."""
    feature = dataset["feature"]
    cases   = dataset["cases"]
    names   = metric_names or FEATURE_DEFAULT_METRICS.get(feature, ["answer_relevancy"])

    print(f"Dataset  : {dataset['name']}")
    print(f"Feature  : {feature}")
    print(f"Metrics  : {', '.join(names)}")
    print(f"Cases    : {len(cases)}\n")

    results: list[dict] = []

    for i, case in enumerate(tqdm(cases, desc="Evaluating", unit="case")):
        # ── Call AI service ──────────────────────────────────────────────────
        ai_error: str | None = None
        try:
            actual_output, latency_ms = call_ai(
                user_input=case["input"],
                feature=feature,
                case=case,
                session_id=f"eval-{uuid4()}",
            )
        except Exception as exc:
            actual_output = f"[AI ERROR: {exc}]"
            latency_ms    = 0
            ai_error      = str(exc)

        # ── Run DeepEval metrics ─────────────────────────────────────────────
        test_case = LLMTestCase(
            input=case["input"],
            actual_output=actual_output,
            expected_output=case.get("expected_output"),
            retrieval_context=case.get("retrieval_context") or None,
        )
        metrics = build_metrics(names, feature=feature)

        try:
            eval_result = evaluate(
                test_cases=[test_case],
                metrics=metrics,
                display_config=DisplayConfig(print_results=False, show_indicator=False),
                cache_config=CacheConfig(write_cache=False, use_cache=False),
            )
            metric_data = (
                eval_result.test_results[0].metrics_data
                if eval_result.test_results
                else []
            )
        except Exception as exc:
            metric_data = []
            ai_error    = ai_error or str(exc)

        results.append({
            "index":          i + 1,
            "input":          case["input"],
            "expected_output": case.get("expected_output", ""),
            "actual_output":  actual_output,
            "latency_ms":     latency_ms,
            "passed":         all(m.success for m in metric_data if m.success is not None),
            "error":          ai_error,
            "metrics": [
                {"name": m.name, "score": m.score, "passed": m.success, "reason": m.reason}
                for m in metric_data
            ],
        })

    return results


def display_results(results: list[dict]) -> None:
    """Render an HTML summary table for a list of evaluation results."""
    total    = len(results)
    passed   = sum(1 for r in results if r["passed"])
    failed   = total - passed
    pass_pct = passed / total * 100 if total else 0

    # Aggregate per-metric stats
    metric_stats: dict[str, dict] = {}
    for r in results:
        for m in r["metrics"]:
            s = metric_stats.setdefault(m["name"], {"total": 0, "sum": 0.0, "passed": 0})
            if m["score"] is not None:
                s["total"] += 1
                s["sum"]   += m["score"]
                if m["passed"]:
                    s["passed"] += 1

    # ── Summary cards ─────────────────────────────────────────────────────────
    def _card(val, label, bg, text_color, label_color):
        return (
            f'<div style="background:{bg};border:1px solid {text_color}30;border-radius:10px;'
            f'padding:14px 22px;text-align:center;min-width:80px">'
            f'<div style="font-size:30px;font-weight:700;color:{text_color}">{val}</div>'
            f'<div style="font-size:10px;color:{label_color};text-transform:uppercase;'
            f'letter-spacing:.1em;margin-top:2px">{label}</div></div>'
        )

    summary = (
        '<div style="font-family:monospace">'
        '<div style="display:flex;gap:16px;margin-bottom:16px">'
        + _card(passed,         "Passed",    "#f0fdf4", "#16a34a", "#4ade80")
        + _card(failed,         "Failed",    "#fef2f2", "#dc2626", "#f87171")
        + _card(total,          "Total",     "#f8fafc", "#1e293b", "#94a3b8")
        + _card(f"{pass_pct:.0f}%", "Pass Rate", "#f8fafc", "#1e293b", "#94a3b8")
        + '</div>'
    )

    if metric_stats:
        summary += (
            '<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;'
            'padding:16px 20px;margin-bottom:16px">'
            '<div style="font-size:10px;color:#94a3b8;text-transform:uppercase;'
            'letter-spacing:.12em;margin-bottom:12px">Metric Summary</div>'
        )
        for name, s in metric_stats.items():
            avg     = s["sum"] / s["total"] if s["total"] else 0
            pct     = int(avg * 100)
            pass_r  = int(s["passed"] / s["total"] * 100) if s["total"] else 0
            color   = "#22c55e" if pct >= 70 else "#f59e0b" if pct >= 50 else "#ef4444"
            summary += (
                '<div style="display:flex;align-items:center;gap:14px;margin-bottom:8px">'
                f'<span style="width:170px;font-size:11px;color:#475569;text-transform:capitalize'
                f';flex-shrink:0">{name.replace("_", " ")}</span>'
                '<div style="flex:0 0 160px;height:6px;background:#e2e8f0;border-radius:9999px;overflow:hidden">'
                f'<div style="height:100%;background:{color};border-radius:9999px;width:{pct}%"></div></div>'
                f'<span style="font-size:11px;font-weight:700;color:{color};width:34px">{pct}%</span>'
                f'<span style="font-size:10px;color:#94a3b8">{pass_r}% pass rate</span>'
                '</div>'
            )
        summary += '</div>'

    summary += '</div>'
    display(HTML(summary))

    # ── Per-case table ─────────────────────────────────────────────────────────
    rows = ""
    for r in results:
        ok       = r["passed"]
        row_bg   = "#f0fdf410" if ok else "#fef2f210"
        s_color  = "#16a34a" if ok else "#dc2626"
        s_label  = "✓ pass" if ok else "✗ fail"

        badges = ""
        for m in r["metrics"]:
            if m["score"] is not None:
                pct = int(m["score"] * 100)
                mc  = "#22c55e" if pct >= 70 else "#f59e0b" if pct >= 50 else "#ef4444"
                badges += (
                    f'<span style="display:inline-block;font-size:9px;font-weight:700;'
                    f'color:{mc};background:{mc}18;border-radius:4px;padding:1px 6px;margin:1px">'
                    f'{m["name"].replace("_", " ")}: {pct}%</span>'
                )
            else:
                badges += (
                    f'<span style="display:inline-block;font-size:9px;color:#94a3b8;'
                    f'background:#f1f5f9;border-radius:4px;padding:1px 6px;margin:1px">'
                    f'{m["name"]}: —</span>'
                )

        reason_html = ""
        for m in r["metrics"]:
            if m.get("reason") and not m["passed"]:
                reason_html += (
                    f'<div style="font-size:9px;color:#94a3b8;margin-top:2px;font-style:italic">'
                    f'{m["name"]}: {m["reason"][:120]}</div>'
                )

        err_html = (
            f'<div style="font-size:9px;color:#dc2626;margin-top:2px">{r["error"]}</div>'
            if r.get("error") else ""
        )

        inp  = str(r["input"]).replace("<", "&lt;").replace(">", "&gt;")
        aout = str(r["actual_output"] or "").replace("<", "&lt;").replace(">", "&gt;")

        rows += (
            f'<tr style="background:{row_bg};border-bottom:1px solid #f1f5f9">'
            f'<td style="padding:8px 12px;font-size:11px;color:#94a3b8;vertical-align:top">#{r["index"]}</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:#334155;max-width:260px;vertical-align:top">'
            f'<div style="max-width:250px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap" title="{inp}">{inp}</div>'
            '</td>'
            f'<td style="padding:8px 12px;font-size:11px;color:#334155;max-width:260px;vertical-align:top">'
            f'<div style="max-width:250px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap" title="{aout}">{aout[:160]}</div>'
            '</td>'
            f'<td style="padding:8px 12px;vertical-align:top">'
            f'<span style="font-size:10px;font-weight:700;color:{s_color}">{s_label}</span>'
            f'<div style="margin-top:3px">{badges}</div>'
            f'{reason_html}{err_html}</td>'
            f'<td style="padding:8px 12px;font-size:10px;color:#94a3b8;vertical-align:top;white-space:nowrap">{r["latency_ms"]}ms</td>'
            '</tr>'
        )

    th = (
        '<th style="padding:8px 12px;text-align:left;font-size:9px;color:#94a3b8;'
        'text-transform:uppercase;letter-spacing:.1em;font-weight:600">'
    )
    table = (
        '<div style="font-family:monospace;margin-top:8px">'
        '<table style="width:100%;border-collapse:collapse">'
        '<thead><tr style="background:#f8fafc;border-bottom:2px solid #e2e8f0">'
        f'{th}#</th>{th}Input</th>{th}Actual Output</th>{th}Result</th>{th}Latency</th>'
        f'</tr></thead><tbody>{rows}</tbody></table></div>'
    )
    display(HTML(table))


print("✓ Setup complete.")

## 3 · Load Datasets

Scans the `datasets/` folder and prints all available test suites.

In [ ]:
DATASETS_DIR = Path("datasets")

datasets: dict[str, dict] = {}
for _path in sorted(DATASETS_DIR.glob("*.json")):
    _d = json.loads(_path.read_text())
    datasets[_d["name"]] = _d

print(f"Found {len(datasets)} dataset(s):\n")
for _name, _d in datasets.items():
    _metrics = FEATURE_DEFAULT_METRICS.get(_d["feature"], ["answer_relevancy"])
    print(f"  {_name}")
    print(f"    feature  : {_d['feature']}")
    print(f"    cases    : {len(_d['cases'])}")
    print(f"    metrics  : {', '.join(_metrics)}")
    print()

## 4 · Evaluate

Set `DATASET_NAME` to one of the keys printed above.  
Set `METRIC_NAMES` to a custom list or leave as `None` to use feature defaults.

In [ ]:
DATASET_NAME = list(datasets.keys())[0]   # change to any dataset name
METRIC_NAMES = None                        # e.g. ["answer_relevancy", "faithfulness"]

results = run_evaluation(datasets[DATASET_NAME], metric_names=METRIC_NAMES)

## 5 · Results

In [ ]:
display_results(results)